#  Notebook 7: Encoding Categorical Variables


In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "City": ["Bengaluru", "Mysuru", "Hassan", "Bengaluru", "Mysuru", "Hassan"],
    "Education": ["School", "College", "PG", "College", "School", "PG"],
    "Subscribed": ["Yes", "No", "Yes", "No", "Yes", "No"],
    "Purchased": [1, 0, 1, 1, 0, 0]
})
print(df)

        City Education Subscribed  Purchased
0  Bengaluru    School        Yes          1
1     Mysuru   College         No          0
2     Hassan        PG        Yes          1
3  Bengaluru   College         No          1
4     Mysuru    School        Yes          0
5     Hassan        PG         No          0


## 1. What is Categorical Encoding?

### Definition
Categorical encoding converts text labels or groups into numbers that machine learning models can process. A category can describe a city, product type, or customer membership. Different encoders preserve different kinds of information. Choosing an encoder depends on the meaning of the categories and the model.

### Business Example
A retail shop records customer cities as names. The names are meaningful to employees but are not numerical measurements. The shop wants to predict purchases using city as an input. Encoding makes that column usable by a model.

### AI/ML Use
Many algorithms require numerical input features. Encoding transforms categorical columns before model training. A suitable method avoids introducing relationships that do not exist. The encoder should be fitted on training data and reused for new data.

### Python Example

In [3]:
print(pd.get_dummies(df[["City"]], dtype=int))

   City_Bengaluru  City_Hassan  City_Mysuru
0               1            0            0
1               0            0            1
2               0            1            0
3               1            0            0
4               0            0            1
5               0            1            0


### Explanation
`get_dummies()` creates a separate indicator for each city. A value of 1 means the row belongs to that city. A value of 0 means it does not. No city is treated as larger than another.

### When to Use / When Not to Use and Risks
Use encoding when a model needs numerical features. Do not encode identifiers without considering whether they carry predictive information. Incorrect encoding can create fake ordering or too many features.

## 2. Nominal Variables

### Definition
Nominal variables are categories without a natural ranking. City, color, and payment method are common examples. One category is not greater than another. Their values are labels rather than quantities.

### Business Example
An online store records payments as UPI, card, or cash. These methods are different choices, not ranked levels. The store compares which methods customers prefer. Numerical labels must not suggest that cash is greater than UPI.

### AI/ML Use
Nominal features can help a model learn patterns between groups. One-hot encoding is often a safe baseline for a manageable number of categories. It avoids assigning an artificial numeric order. Tree-based models may have additional category-handling options.

### Python Example

In [5]:
print(pd.get_dummies(df["City"], prefix="City", dtype=int))

   City_Bengaluru  City_Hassan  City_Mysuru
0               1            0            0
1               0            0            1
2               0            1            0
3               1            0            0
4               0            0            1
5               0            1            0


### Explanation
Each city receives its own binary column. The rows show exactly which city applies. There is no assumed distance between cities. This suits nominal data with relatively few categories.

### When to Use / When Not to Use and Risks
Use one-hot encoding for low-cardinality nominal features. Avoid plain integer codes in distance-based or linear models because they imply an order.

## 3. Ordinal Variables

### Definition
Ordinal variables have a meaningful order, such as low, medium, and high. The ranks are known, but the gaps between ranks may not be equal. The correct order should come from domain knowledge. A numeric encoding can preserve that sequence.

### Business Example
A business classifies service priority as Low, Medium, or High. High-priority requests should be handled first. The labels represent a genuine ranking. The business needs the model to preserve that order.

### AI/ML Use
Ordinal encoding gives models a compact representation of ranked groups. The assigned numbers communicate relative order. The model may also interpret numeric gaps as meaningful, so this assumption needs checking. Ordinal encoding is unsuitable for unordered categories.

### Python Example

In [7]:
priority = pd.Series(["Low", "High", "Medium", "Low"])
print(priority.map({"Low": 0, "Medium": 1, "High": 2}))

0    0
1    2
2    1
3    0
dtype: int64


### Explanation
The mapping gives Low 0, Medium 1, and High 2. The original rank is preserved. It does not prove that the jump from Low to Medium equals the jump from Medium to High. Use a domain-approved ordering.

### When to Use / When Not to Use and Risks
Use for genuinely ordered groups. Do not apply an arbitrary rank to cities, colors, or other nominal variables.

## 4. Binary Variables

### Definition
Binary variables contain exactly two possible categories. Examples include Yes/No and Active/Inactive. They can usually be represented with 0 and 1. The chosen mapping should be documented.

### Business Example
A subscription company stores whether each customer subscribed. The values are Yes or No. The company wants to predict subscription behavior. A single binary column is sufficient.

### AI/ML Use
Binary encoding uses one feature rather than multiple dummy columns. It is easy for models to process and interpret. The meaning of 0 and 1 must remain consistent between training and prediction. Missing or unexpected values require separate handling.

### Python Example

In [9]:
print(df["Subscribed"].map({"No": 0, "Yes": 1}))

0    1
1    0
2    1
3    0
4    1
5    0
Name: Subscribed, dtype: int64


### Explanation
No becomes 0 and Yes becomes 1. Each row keeps its original meaning. The result can be used as a numeric feature. Unexpected values would become missing and should be checked.

### When to Use / When Not to Use and Risks
Use when exactly two categories exist. Do not silently map unknown or missing values to No.

## 5. Label Encoding

### Definition
Label encoding assigns a distinct integer to each class label. It is commonly used for a classification target. For example, class names can become 0, 1, and 2. As an input feature, the numbers may wrongly suggest an order.

### Business Example
A customer support team labels ticket outcomes as Resolved or Escalated. A classifier needs a numeric target for training. The team can convert those target labels to integers. The label-to-number mapping should be saved.

### AI/ML Use
`LabelEncoder` is designed mainly for target labels. It produces compact class IDs for classification. Applying it to nominal input features may introduce artificial ranking. New target labels at prediction time also require careful handling.

### Python Example

In [11]:
from sklearn.preprocessing import LabelEncoder
labels = pd.Series(["Resolved", "Escalated", "Resolved"])
encoder = LabelEncoder()
print(encoder.fit_transform(labels))
print(dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))

[1 0 1]
{'Escalated': 0, 'Resolved': 1}


### Explanation
The encoder learns the two class names and assigns numeric IDs. The mapping is printed so the values can be interpreted. This example encodes a target rather than an input feature. For nominal inputs, prefer one-hot encoding or a suitable alternative.

### When to Use / When Not to Use and Risks
Use primarily for classification targets. Avoid using label IDs as meaningful numeric input values in linear or distance-based models.

## 6. Ordinal Encoding

### Definition
Ordinal encoding assigns numeric codes to ordered categories. The sequence must be specified correctly. It can be implemented with a dictionary or an ordinal encoder. Unknown values should not be assigned an accidental rank.

### Business Example
A company rates customer satisfaction as Poor, Average, and Excellent. The levels have a clear order. The company wants to use satisfaction as a model input. Ordered codes preserve that ranking.

### AI/ML Use
Ordinal encoding can work well for ranked survey responses. It creates one column instead of many. Models may assume the numeric differences are equally spaced. That assumption should be assessed for the chosen model.

### Python Example

In [13]:
from sklearn.preprocessing import OrdinalEncoder
levels = pd.DataFrame({"Rating": ["Poor", "Excellent", "Average"]})
enc = OrdinalEncoder(categories=[["Poor", "Average", "Excellent"]])
print(enc.fit_transform(levels))

[[0.]
 [2.]
 [1.]]


### Explanation
The specified list defines the rank from Poor to Excellent. The output codes are 0, 2, and 1 in row order. Explicit ordering prevents alphabetical ranking mistakes. Fit the encoder on training data only in a real ML workflow.

### When to Use / When Not to Use and Risks
Use for ordered categories. Do not use for unordered features or assume the numeric distances are exact.

## 7. One-Hot Encoding

### Definition
One-hot encoding creates one indicator column per category. Each row marks its category with 1 and the others with 0. It does not impose a ranking between categories. The number of columns grows with the number of unique categories.

### Business Example
A delivery company operates in three cities. It wants to predict delivery delays. Each city can have a different effect on delay. Separate indicator columns let the model learn those effects.

### AI/ML Use
One-hot encoding is useful for nominal input variables. It works well with many linear and distance-based models. The same learned columns must be used for test and future data. Unseen categories need an explicit handling policy.

### Python Example

In [15]:
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
result = enc.fit_transform(df[["City"]])
print(pd.DataFrame(result, columns=enc.get_feature_names_out()))

   City_Bengaluru  City_Hassan  City_Mysuru
0             1.0          0.0          0.0
1             0.0          0.0          1.0
2             0.0          1.0          0.0
3             1.0          0.0          0.0
4             0.0          0.0          1.0
5             0.0          1.0          0.0


### Explanation
The encoder produces one column for each city found during fitting. Each row has 1 in the matching city column. `handle_unknown="ignore"` prevents errors for unseen cities during transformation. In a real project, fit the encoder using training data only.

### When to Use / When Not to Use and Risks
Use for nominal features with manageable unique values. Avoid it for extremely high-cardinality columns without controlling feature growth.

## 8. Dummy Variables

### Definition
Dummy variables are 0/1 columns representing categories. They are commonly created with Pandas `get_dummies()`. Sometimes one category is dropped to act as a reference group. This can be useful in linear regression with an intercept.

### Business Example
A shop records product categories as Food, Clothing, and Electronics. It wants to compare sales patterns across categories. Dummy columns represent these groups numerically. One group can serve as the reference category.

### AI/ML Use
Dummy variables are a form of categorical encoding. Dropping one category can reduce redundant columns in certain linear models. Keeping all columns is often acceptable for tree-based models. The choice depends on the model and preprocessing pipeline.

### Python Example

In [17]:
print(pd.get_dummies(df["City"], drop_first=True, dtype=int))

   Hassan  Mysuru
0       0       0
1       0       1
2       1       0
3       0       0
4       0       1
5       1       0


### Explanation
`drop_first=True` removes one city indicator column. The removed city becomes the reference group. Rows with zeros in the remaining city columns belong to that reference group. This is not required for every model.

### When to Use / When Not to Use and Risks
Use dummy variables for nominal data; consider dropping a reference category for linear models. Avoid assuming dropped categories are unimportant.

## 9. Frequency Encoding

### Definition
Frequency encoding replaces a category with how often it appears. Counts or proportions can be used. It produces a single numeric column even with many categories. Different categories can share the same frequency.

### Business Example
An e-commerce platform has thousands of seller names. One-hot encoding would create too many columns. The platform can use seller occurrence frequency as a compact feature. Popular and rare sellers then receive different values.

### AI/ML Use
Frequency encoding can reduce feature count for high-cardinality data. It does not directly use the target, so it avoids target-based leakage. Frequency may still capture dataset-specific patterns that change over time. Learn frequencies from training data and handle unseen categories.

### Python Example

In [19]:
counts = df["City"].value_counts()
print(df["City"].map(counts))

0    2
1    2
2    2
3    2
4    2
5    2
Name: City, dtype: int64


### Explanation
`value_counts()` counts the number of rows for each city. `map()` replaces each city name with its count. In this sample, every city appears twice, so all receive 2. This shows a limitation: equally frequent categories become indistinguishable.

### When to Use / When Not to Use and Risks
Use when many categories make one-hot encoding expensive. Avoid when category identity matters more than its frequency; fit counts on training data only.

## 10. Target Encoding

### Definition
Target encoding replaces each category with a statistic of the target, often its mean. It can summarize category-level predictive information. It is especially useful for high-cardinality features. However, computing it on the same rows used for training can cause target leakage and overfitting.

### Business Example
A retailer wants to predict whether customers will purchase. It has many store locations. Each location has a different historical purchase rate. A carefully calculated purchase rate can serve as an encoded feature.

### AI/ML Use
Target encoding can help models learn category effects without thousands of columns. It must be calculated using training data only. For training rows, use out-of-fold or cross-fitted encoding to avoid leakage. Smoothing and handling unseen categories can improve stability.

### Python Example

In [21]:
from sklearn.preprocessing import TargetEncoder
X = df[["City"]]
y = df["Purchased"]
te = TargetEncoder(random_state=42, cv=3)
encoded_train = te.fit_transform(X, y)  # Cross-fitted training values
print(pd.DataFrame(encoded_train, columns=["City_encoded"]).round(3))

   City_encoded
0           1.0
1           0.0
2           0.0
3           1.0
4           0.0
5           1.0


### Explanation
`TargetEncoder` learns purchase-rate information from the training labels. `fit_transform()` uses cross-fitting to reduce training leakage. Encoded values can vary by fold and are not simple raw category means. For new data, use the fitted encoder’s `transform()` method; never use its target values.

### When to Use / When Not to Use and Risks
Use for predictive high-cardinality categories with enough data. Do not compute target means using validation/test targets or the same row’s label.

## 11. Handling Unknown Categories

### Definition
Unknown categories are values that were not present when an encoder was fitted. They often appear in validation, test, or production data. An encoder should have a predictable policy for these values. Silently assigning an arbitrary existing category can mislead a model.

### Business Example
A delivery app trains on Bengaluru and Mysuru orders. Later, it receives an order from Hassan. The model has never seen Hassan during training. The preprocessing step must still process the new order safely.

### AI/ML Use
Robust preprocessing is important when models are deployed. `OneHotEncoder(handle_unknown="ignore")` produces zeros for unseen categories. Other encoders may use a dedicated unknown code or a fallback value. The same fitted encoder must be reused after deployment.

### Python Example

In [23]:
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
enc.fit(pd.DataFrame({"City": ["Bengaluru", "Mysuru"]}))
print(enc.transform(pd.DataFrame({"City": ["Hassan"]})))

[[0. 0.]]


### Explanation
The encoder learns only Bengaluru and Mysuru. Hassan is new, so the output is all zeros rather than an error. This keeps the feature shape consistent. Monitor unknown-category rates because frequent new categories may require retraining.

### When to Use / When Not to Use and Risks
Use explicit unknown-category handling for production models. Avoid refitting an encoder separately on test or prediction data.

## 12. High-Cardinality Categories

### Definition
A high-cardinality categorical feature has many distinct values. Examples include product IDs, seller names, and postal codes. One-hot encoding such columns can create very wide datasets. Some high-cardinality fields are identifiers with little useful predictive meaning.

### Business Example
A marketplace has thousands of unique product codes. Creating a column for every code can be expensive. The company needs to decide whether product identity helps prediction. It may group rare categories or choose a compact encoder.

### AI/ML Use
High-cardinality inputs can increase memory use and overfitting risk. Frequency encoding, carefully cross-fitted target encoding, hashing, or rare-category grouping are alternatives. The best choice depends on data volume and the model. Validation performance and leakage checks should guide the decision.

### Python Example

In [25]:
products = pd.Series(["P1", "P2", "P3", "P1", "P4", "P5"])
counts = products.value_counts()
print(products.where(products.map(counts) >= 2, "Other"))

0       P1
1    Other
2    Other
3       P1
4    Other
5    Other
dtype: object


### Explanation
`value_counts()` finds how often each product appears. Categories appearing fewer than two times become Other. P1 remains because it appears twice. Grouping rare values reduces the number of distinct categories, but it can also hide useful differences.

### When to Use / When Not to Use and Risks
Use rare-category grouping or compact encoding when unique values are numerous. Avoid treating unique row IDs as meaningful predictors without evidence.